In [13]:
from pathlib import Path
root_dir = Path("/Users/jliu/workspace/ICL")
data_dir = root_dir / "datasets"
model_dir = root_dir / "models"
result_dir = root_dir / "results"

# Core data schemas

In [4]:
"""Data schemas for comprehensive evaluation results."""

from dataclasses import dataclass, field
from pathlib import Path
from datetime import datetime
import typing as t
import pandas as pd
import numpy as np

# Type aliases
ConfigTuple = tuple[int, int]
TransferCondition = t.Literal["within_config", "cross_L", "cross_m", "cross_config"]
ControlType = t.Literal["normal", "shuffled_context", "random_context"]

@dataclass
class ModelMetadata:
    """Metadata for a single model checkpoint."""
    model_id: str
    config_L: int
    config_m: int
    n_train: int
    checkpoint_step: int
    checkpoint_path: Path
    model_type: str  # "causal_lm" or "mlm"
    training_seed: int | None = None
    eval_seed: int | None = None
    
    def to_dict(self) -> dict[str, t.Any]:
        """Convert to dictionary for serialization."""
        return {
            "model_id": self.model_id,
            "config_L": self.config_L,
            "config_m": self.config_m,
            "n_train": self.n_train,
            "checkpoint_step": self.checkpoint_step,
            "checkpoint_path": str(self.checkpoint_path),
            "model_type": self.model_type,
            "training_seed": self.training_seed,
            "eval_seed": self.eval_seed,
        }

@dataclass
class ICLPerformanceRecord:
    """Single ICL evaluation result record."""
    model_id: str
    config_L: int
    config_m: int
    n_train: int
    checkpoint_step: int
    context_size: int
    transfer_condition: TransferCondition
    target_config_L: int
    target_config_m: int
    accuracy: float
    sequence_id: int
    control_type: ControlType
    evaluation_timestamp: datetime
    num_sequences: int = 0
    num_correct: int = 0
    
    def to_dict(self) -> dict[str, t.Any]:
        """Convert to dictionary for DataFrame creation."""
        return {
            "model_id": self.model_id,
            "config_L": self.config_L,
            "config_m": self.config_m,
            "n_train": self.n_train,
            "checkpoint_step": self.checkpoint_step,
            "context_size": self.context_size,
            "transfer_condition": self.transfer_condition,
            "target_config_L": self.target_config_L,
            "target_config_m": self.target_config_m,
            "accuracy": self.accuracy,
            "sequence_id": self.sequence_id,
            "control_type": self.control_type,
            "evaluation_timestamp": self.evaluation_timestamp,
            "num_sequences": self.num_sequences,
            "num_correct": self.num_correct,
        }

@dataclass
class AttentionRecord:
    """Single attention pattern record."""
    model_id: str
    layer_idx: int
    head_idx: int
    context_size: int
    sequence_id: int
    attention_matrix: np.ndarray
    evaluation_timestamp: datetime
    
    def get_filename(self) -> str:
        """Generate filename for attention data."""
        return f"{self.model_id}_layer{self.layer_idx}_head{self.head_idx}_k{self.context_size}_seq{self.sequence_id}.npz"

@dataclass
class EvaluationConfig:
    """Configuration for comprehensive evaluation."""
    # Model and data paths
    checkpoint_base_dirs: list[Path]
    eval_dataset_path: Path
    output_dir: Path
    
    # Evaluation parameters
    context_sizes: list[int] = field(default_factory=lambda: [1, 2, 3, 4, 5, 6, 8])
    transfer_conditions: list[TransferCondition] = field(
        default_factory=lambda: ["within_config", "cross_L", "cross_m", "cross_config"]
    )
    control_types: list[ControlType] = field(
        default_factory=lambda: ["normal", "shuffled_context", "random_context"]
    )
    
    # Model configurations to evaluate
    target_configs: list[ConfigTuple] = field(default_factory=list)
    diversity_levels: list[int] = field(default_factory=lambda: [8, 16, 32, 64, 128])
    model_types: list[str] = field(default_factory=lambda: ["causal_lm", "mlm"])
    
    # Computational parameters
    device: str = "cuda"
    batch_size: int = 32
    max_sequences_per_condition: int = 200
    capture_attention: bool = True
    capture_representations: bool = False
    
    # Output control
    save_intermediate: bool = True
    overwrite_existing: bool = False
    
    def validate(self) -> bool:
        """Validate configuration parameters."""
        # Check paths exist
        for checkpoint_dir in self.checkpoint_base_dirs:
            if not checkpoint_dir.exists():
                raise FileNotFoundError(f"Checkpoint directory not found: {checkpoint_dir}")
        
        if not self.eval_dataset_path.exists():
            raise FileNotFoundError(f"Evaluation dataset not found: {self.eval_dataset_path}")
        
        # Validate parameters
        if not self.context_sizes or any(k <= 0 for k in self.context_sizes):
            raise ValueError("Context sizes must be positive integers")
        
        if not self.target_configs:
            raise ValueError("Must specify target configurations to evaluate")
        
        return True

class DataSchemaManager:
    """Manages data schemas and DataFrame operations."""
    
    @staticmethod
    def create_icl_performance_schema() -> dict[str, str]:
        """Define ICL performance DataFrame schema."""
        return {
            "model_id": "string",
            "config_L": "int32",
            "config_m": "int32", 
            "n_train": "int32",
            "checkpoint_step": "int32",
            "context_size": "int32",
            "transfer_condition": "category",
            "target_config_L": "int32",
            "target_config_m": "int32",
            "accuracy": "float64",
            "sequence_id": "int32",
            "control_type": "category",
            "evaluation_timestamp": "datetime64[ns]",
            "num_sequences": "int32",
            "num_correct": "int32",
        }
    
    @staticmethod
    def create_model_metadata_schema() -> dict[str, str]:
        """Define model metadata DataFrame schema."""
        return {
            "model_id": "string",
            "config_L": "int32",
            "config_m": "int32",
            "n_train": "int32", 
            "checkpoint_step": "int32",
            "checkpoint_path": "string",
            "model_type": "category",
            "training_seed": "Int32",  # Nullable integer
            "eval_seed": "Int32",      # Nullable integer
        }
    
    @staticmethod
    def records_to_dataframe(
        records: list[ICLPerformanceRecord]
    ) -> pd.DataFrame:
        """Convert ICL performance records to typed DataFrame."""
        if not records:
            # Return empty DataFrame with correct schema
            schema = DataSchemaManager.create_icl_performance_schema()
            return pd.DataFrame().astype(schema)
        
        data = [record.to_dict() for record in records]
        df = pd.DataFrame(data)
        
        # Apply schema
        schema = DataSchemaManager.create_icl_performance_schema()
        for col, dtype in schema.items():
            if col in df.columns:
                if dtype == "category":
                    df[col] = df[col].astype(dtype)
                else:
                    df[col] = df[col].astype(dtype)
        
        return df
    
    @staticmethod
    def metadata_to_dataframe(
        metadata: list[ModelMetadata]
    ) -> pd.DataFrame:
        """Convert model metadata to typed DataFrame."""
        if not metadata:
            schema = DataSchemaManager.create_model_metadata_schema()
            return pd.DataFrame().astype(schema)
        
        data = [meta.to_dict() for meta in metadata]
        df = pd.DataFrame(data)
        
        # Apply schema
        schema = DataSchemaManager.create_model_metadata_schema()
        for col, dtype in schema.items():
            if col in df.columns:
                df[col] = df[col].astype(dtype)
        
        return df

def create_evaluation_manifest(
    config: EvaluationConfig,
    start_time: datetime,
    end_time: datetime | None = None,
    status: str = "running"
) -> dict[str, t.Any]:
    """Create evaluation run manifest."""
    return {
        "experiment_id": f"comprehensive_eval_{start_time.strftime('%Y%m%d_%H%M%S')}",
        "start_time": start_time.isoformat(),
        "end_time": end_time.isoformat() if end_time else None,
        "status": status,
        "config": {
            "context_sizes": config.context_sizes,
            "transfer_conditions": config.transfer_conditions,
            "control_types": config.control_types,
            "target_configs": config.target_configs,
            "diversity_levels": config.diversity_levels,
            "model_types": config.model_types,
            "device": config.device,
            "batch_size": config.batch_size,
            "max_sequences_per_condition": config.max_sequences_per_condition,
            "capture_attention": config.capture_attention,
            "capture_representations": config.capture_representations,
        },
        "data_schema_version": "1.0",
        "output_files": {
            "icl_performance": "raw_evaluations/icl_performance.parquet",
            "model_registry": "metadata/model_registry.parquet", 
            "attention_data": "raw_evaluations/attention_data/",
            "intermediate_metrics": "intermediate/aggregated_metrics.parquet",
        }
    }

# Checkpoint Manager

In [10]:
"""Checkpoint management and model loading utilities."""


import json
import re
import warnings
from collections import defaultdict
import typing as t

import torch
from transformers import AutoModelForCausalLM, AutoModelForMaskedLM, AutoTokenizer
import pandas as pd


class CheckpointManager:
    """Manages model checkpoint discovery, loading, and metadata extraction."""
    
    def __init__(self, config: EvaluationConfig):
        """Initialize checkpoint manager with evaluation configuration."""
        self.config = config
        self.device = torch.device(config.device)
        self._checkpoint_cache: dict[str, tuple[t.Any, t.Any]] = {}
        
    def discover_checkpoints(self) -> list[ModelMetadata]:
        """Discover all available checkpoints matching target configurations."""
        all_metadata = []
        
        for checkpoint_dir in self.config.checkpoint_base_dirs:
            metadata = self._scan_checkpoint_directory(checkpoint_dir)
            all_metadata.extend(metadata)
        
        # Filter by target configurations and diversity levels
        filtered_metadata = self._filter_by_targets(all_metadata)
        
        print(f"Discovered {len(filtered_metadata)} checkpoints across {len(self.config.checkpoint_base_dirs)} directories")
        self._print_discovery_summary(filtered_metadata)
        
        return filtered_metadata
    
    def _scan_checkpoint_directory(self, checkpoint_dir: Path) -> list[ModelMetadata]:
        """Scan a single checkpoint directory for model files."""
        metadata_list = []
        
        # Look for standard checkpoint patterns
        checkpoint_patterns = [
            "*/checkpoint-*/",           # Hugging Face standard
            "*/step_*/",                 # Custom step naming
            "*/*_L*_m*_ntrain*/",       # Config-based naming
        ]
        
        found_checkpoints = set()
        for pattern in checkpoint_patterns:
            found_checkpoints.update(checkpoint_dir.glob(pattern))
        
        for checkpoint_path in found_checkpoints:
            if self._is_valid_checkpoint(checkpoint_path):
                try:
                    metadata = self._extract_checkpoint_metadata(checkpoint_path)
                    if metadata:
                        metadata_list.append(metadata)
                except Exception as e:
                    warnings.warn(f"Failed to process checkpoint {checkpoint_path}: {e}")
        
        return metadata_list
    
    def _is_valid_checkpoint(self, checkpoint_path: Path) -> bool:
        """Check if path contains a valid model checkpoint."""
        required_files = ["config.json"]
        model_files = ["pytorch_model.bin", "model.safetensors", "model.pt"]
        
        # Check for config file
        if not any((checkpoint_path / f).exists() for f in required_files):
            return False
        
        # Check for model weights
        if not any((checkpoint_path / f).exists() for f in model_files):
            return False
        
        return True
    
    def _extract_checkpoint_metadata(self, checkpoint_path: Path) -> ModelMetadata | None:
        """Extract metadata from checkpoint path and config files."""
        # Try to extract config info from path structure
        config_info = self._parse_checkpoint_path(checkpoint_path)
        
        # Try to load config.json for additional info
        config_json_path = checkpoint_path / "config.json"
        if config_json_path.exists():
            try:
                with open(config_json_path) as f:
                    model_config = json.load(f)
                config_info.update(self._parse_model_config(model_config))
            except Exception as e:
                warnings.warn(f"Failed to parse config.json at {config_json_path}: {e}")
        
        # Try to extract training info from training_args.json
        training_args_path = checkpoint_path / "training_args.json"
        if training_args_path.exists():
            try:
                with open(training_args_path) as f:
                    training_args = json.load(f)
                config_info.update(self._parse_training_args(training_args))
            except Exception:
                pass  # Optional file
        
        # Validate required fields
        required_fields = ["config_L", "config_m", "n_train", "checkpoint_step", "model_type"]
        if not all(field in config_info for field in required_fields):
            warnings.warn(f"Missing required metadata fields for {checkpoint_path}")
            return None
        
        # Generate model ID
        model_id = self._generate_model_id(config_info, checkpoint_path)
        
        return ModelMetadata(
            model_id=model_id,
            config_L=config_info["config_L"],
            config_m=config_info["config_m"],
            n_train=config_info["n_train"],
            checkpoint_step=config_info["checkpoint_step"],
            checkpoint_path=checkpoint_path,
            model_type=config_info["model_type"],
            training_seed=config_info.get("training_seed"),
            eval_seed=config_info.get("eval_seed"),
        )
    
    def _parse_checkpoint_path(self, checkpoint_path: Path) -> dict[str, t.Any]:
        """Extract configuration info from checkpoint path structure."""
        path_str = str(checkpoint_path)
        config_info = {}
        
        # Extract L and m from path patterns like "L2_m3" or "config_L2_m3"
        config_pattern = r"(?:config_)?L(\d+)_m(\d+)"
        config_match = re.search(config_pattern, path_str)
        if config_match:
            config_info["config_L"] = int(config_match.group(1))
            config_info["config_m"] = int(config_match.group(2))
        
        # Extract N_train from patterns like "ntrain64" or "n_train_128"
        ntrain_pattern = r"n?_?train[_-]?(\d+)"
        ntrain_match = re.search(ntrain_pattern, path_str, re.IGNORECASE)
        if ntrain_match:
            config_info["n_train"] = int(ntrain_match.group(1))
        
        # Extract checkpoint step from patterns like "checkpoint-1000" or "step_1000"
        step_patterns = [
            r"checkpoint[_-](\d+)",
            r"step[_-](\d+)",
            r"(\d+)_steps",
        ]
        for pattern in step_patterns:
            step_match = re.search(pattern, path_str)
            if step_match:
                config_info["checkpoint_step"] = int(step_match.group(1))
                break
        
        # Infer model type from path
        if "causal" in path_str.lower() or "clm" in path_str.lower():
            config_info["model_type"] = "causal_lm"
        elif "masked" in path_str.lower() or "mlm" in path_str.lower():
            config_info["model_type"] = "mlm"
        
        return config_info
    
    def _parse_model_config(self, model_config: dict) -> dict[str, t.Any]:
        """Extract relevant info from model config.json."""
        config_info = {}
        
        # Try to infer model type from architecture
        if "architectures" in model_config:
            arch = model_config["architectures"][0].lower()
            if "causallm" in arch or "gpt" in arch:
                config_info["model_type"] = "causal_lm"
            elif "maskedlm" in arch or "bert" in arch:
                config_info["model_type"] = "mlm"
        
        return config_info
    
    def _parse_training_args(self, training_args: dict) -> dict[str, t.Any]:
        """Extract relevant info from training_args.json."""
        config_info = {}
        
        if "seed" in training_args:
            config_info["training_seed"] = training_args["seed"]
        
        if "eval_steps" in training_args:
            config_info["eval_steps"] = training_args["eval_steps"]
        
        return config_info
    
    def _generate_model_id(self, config_info: dict, checkpoint_path: Path) -> str:
        """Generate unique model identifier."""
        components = [
            config_info["model_type"],
            f"L{config_info['config_L']}",
            f"m{config_info['config_m']}",
            f"ntrain{config_info['n_train']}",
            f"step{config_info['checkpoint_step']}",
        ]
        
        # Add path hash for uniqueness
        path_hash = str(abs(hash(str(checkpoint_path))))[-6:]
        components.append(path_hash)
        
        return "_".join(components)
    
    def _filter_by_targets(self, metadata_list: list[ModelMetadata]) -> list[ModelMetadata]:
        """Filter checkpoints by target configurations and diversity levels."""
        if not self.config.target_configs:
            return metadata_list
        
        filtered = []
        for metadata in metadata_list:
            config = (metadata.config_L, metadata.config_m)
            
            # Check if config matches targets
            if config in self.config.target_configs:
                # Check if diversity level matches
                if metadata.n_train in self.config.diversity_levels:
                    # Check if model type matches
                    if metadata.model_type in self.config.model_types:
                        filtered.append(metadata)
        
        return filtered
    
    def _print_discovery_summary(self, metadata_list: list[ModelMetadata]) -> None:
        """Print summary of discovered checkpoints."""
        if not metadata_list:
            print("No checkpoints found matching criteria")
            return
        
        # Group by configuration
        by_config = defaultdict(list)
        for meta in metadata_list:
            key = (meta.config_L, meta.config_m, meta.n_train, meta.model_type)
            by_config[key].append(meta)
        
        print("\nCheckpoint Discovery Summary:")
        print("-" * 50)
        for (L, m, n_train, model_type), metas in sorted(by_config.items()):
            steps = [meta.checkpoint_step for meta in metas]
            print(f"L={L}, m={m}, N_train={n_train}, {model_type}: {len(steps)} checkpoints (steps: {min(steps)}-{max(steps)})")
    
    def load_model_checkpoint(
        self, 
        metadata: ModelMetadata, 
        cache: bool = True
    ) -> tuple[t.Any, t.Any]:
        """Load model and tokenizer from checkpoint."""
        if cache and metadata.model_id in self._checkpoint_cache:
            return self._checkpoint_cache[metadata.model_id]
        
        print(f"Loading checkpoint: {metadata.model_id}")
        
        try:
            # Load tokenizer
            tokenizer = AutoTokenizer.from_pretrained(
                metadata.checkpoint_path,
                trust_remote_code=True
            )
            
            # Ensure tokenizer has pad token
            if tokenizer.pad_token is None:
                if tokenizer.eos_token is not None:
                    tokenizer.pad_token = tokenizer.eos_token
                else:
                    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            
            # Load model based on type
            if metadata.model_type == "causal_lm":
                model = AutoModelForCausalLM.from_pretrained(
                    metadata.checkpoint_path,
                    trust_remote_code=True,
                    torch_dtype=torch.float16 if self.device.type == "cuda" else torch.float32,
                )
            elif metadata.model_type == "mlm":
                model = AutoModelForMaskedLM.from_pretrained(
                    metadata.checkpoint_path,
                    trust_remote_code=True,
                    torch_dtype=torch.float16 if self.device.type == "cuda" else torch.float32,
                )
            else:
                raise ValueError(f"Unsupported model type: {metadata.model_type}")
            
            # Move to device
            model = model.to(self.device)
            model.eval()
            
            # Resize embeddings if tokenizer was modified
            if len(tokenizer) != model.get_input_embeddings().num_embeddings:
                model.resize_token_embeddings(len(tokenizer))
            
            if cache:
                self._checkpoint_cache[metadata.model_id] = (model, tokenizer)
            
            return model, tokenizer
            
        except Exception as e:
            raise RuntimeError(f"Failed to load checkpoint {metadata.checkpoint_path}: {e}")
    
    def validate_checkpoint_completeness(
        self, 
        metadata_list: list[ModelMetadata]
    ) -> dict[str, t.Any]:
        """Validate that we have complete checkpoint coverage."""
        validation_results = {
            "total_checkpoints": len(metadata_list),
            "configs_found": set(),
            "missing_configs": [],
            "diversity_coverage": defaultdict(set),
            "model_type_coverage": defaultdict(set),
        }
        
        # Analyze what we have
        for meta in metadata_list:
            config = (meta.config_L, meta.config_m)
            validation_results["configs_found"].add(config)
            validation_results["diversity_coverage"][config].add(meta.n_train)
            validation_results["model_type_coverage"][config].add(meta.model_type)
        
        # Check for missing target configs
        for target_config in self.config.target_configs:
            if target_config not in validation_results["configs_found"]:
                validation_results["missing_configs"].append(target_config)
        
        # Check diversity coverage
        incomplete_diversity = []
        for config in validation_results["configs_found"]:
            found_diversity = validation_results["diversity_coverage"][config]
            expected_diversity = set(self.config.diversity_levels)
            if not expected_diversity.issubset(found_diversity):
                missing = expected_diversity - found_diversity
                incomplete_diversity.append((config, list(missing)))
        
        validation_results["incomplete_diversity"] = incomplete_diversity
        
        return validation_results
    
    def clear_cache(self) -> None:
        """Clear the model cache to free memory."""
        for model, tokenizer in self._checkpoint_cache.values():
            if hasattr(model, "cpu"):
                model.cpu()
            del model, tokenizer
        
        self._checkpoint_cache.clear()
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Comprehensive Evaluator

In [11]:
"""Comprehensive evaluation pipeline for all ICL experiments."""


class ICLEvaluationEngine:
    """Core ICL evaluation functionality."""
    
    def __init__(self, device: str = "cuda"):
        """Initialize evaluation engine."""
        self.device = torch.device(device)
    
    def evaluate_icl_sequence(
        self,
        model: t.Any,
        tokenizer: t.Any,
        sequence: dict[str, t.Any],
        capture_attention: bool = False
    ) -> tuple[bool, dict[str, t.Any] | None]:
        """Evaluate a single ICL sequence and optionally capture attention."""
        try:
            # Prepare context and query
            context_features = sequence["context_features"]
            context_labels = sequence["context_labels"] 
            query_features = sequence["query_features"]
            true_label = sequence["query_label"]
            
            # Format as ICL prompt
            prompt = self._format_icl_prompt(
                context_features, context_labels, query_features, tokenizer
            )
            
            # Tokenize
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(self.device)
            
            # Get predictions and attention
            with torch.no_grad():
                if capture_attention:
                    outputs = model(**inputs, output_attentions=True)
                    attention_data = self._extract_attention_patterns(outputs.attentions)
                else:
                    outputs = model(**inputs)
                    attention_data = None
                
                # Get prediction
                if hasattr(outputs, "logits"):
                    logits = outputs.logits
                else:
                    logits = outputs
                
                predicted_label = self._extract_prediction(logits, tokenizer, true_label)
                is_correct = predicted_label == true_label
            
            return is_correct, attention_data
            
        except Exception as e:
            warnings.warn(f"Evaluation failed for sequence: {e}")
            return False, None
    
    def _format_icl_prompt(
        self,
        context_features: list[list[int]],
        context_labels: list[int],
        query_features: list[int],
        tokenizer: t.Any
    ) -> str:
        """Format features and labels as ICL prompt."""
        prompt_parts = []
        
        # Add context examples
        for features, label in zip(context_features, context_labels, strict=False):
            feature_str = " ".join(map(str, features))
            prompt_parts.append(f"Input: {feature_str} Output: {label}")
        
        # Add query
        query_str = " ".join(map(str, query_features))
        prompt_parts.append(f"Input: {query_str} Output:")
        
        return "\n".join(prompt_parts)
    
    def _extract_prediction(
        self,
        logits: torch.Tensor,
        tokenizer: t.Any,
        true_label: int
    ) -> int:
        """Extract predicted label from model logits."""
        # Get logits for the last token (where prediction should be)
        last_token_logits = logits[0, -1, :]
        
        # Get top prediction
        predicted_token_id = torch.argmax(last_token_logits).item()
        predicted_token = tokenizer.decode([predicted_token_id]).strip()
        
        # Try to convert to integer
        try:
            predicted_label = int(predicted_token)
            return predicted_label
        except ValueError:
            # If can't convert, try to extract digit
            import re
            digits = re.findall(r'\d+', predicted_token)
            if digits:
                return int(digits[0])
            else:
                # Return random guess if no valid prediction
                return -1
    
    def _extract_attention_patterns(
        self, 
        attention_tensors: tuple[torch.Tensor, ...]
    ) -> dict[str, t.Any]:
        """Extract and process attention patterns."""
        attention_data = {}
        
        for layer_idx, layer_attention in enumerate(attention_tensors):
            # layer_attention shape: [batch_size, num_heads, seq_len, seq_len]
            layer_attention = layer_attention.squeeze(0)  # Remove batch dimension
            
            for head_idx in range(layer_attention.size(0)):
                head_attention = layer_attention[head_idx].cpu().numpy()
                key = f"layer_{layer_idx}_head_{head_idx}"
                attention_data[key] = head_attention
        
        return attention_data

class ComprehensiveEvaluator:
    """Main comprehensive evaluation pipeline."""
    
    def __init__(self, config: EvaluationConfig):
        """Initialize comprehensive evaluator."""
        self.config = config
        self.config.validate()
        
        self.checkpoint_manager = CheckpointManager(config)
        self.evaluation_engine = ICLEvaluationEngine(config.device)
        self.evaluation_dataset = None
        self._setup_output_directories()
    
    def _setup_output_directories(self) -> None:
        """Create necessary output directories."""
        directories = [
            self.config.output_dir / "metadata",
            self.config.output_dir / "raw_evaluations",
            self.config.output_dir / "raw_evaluations" / "attention_data",
            self.config.output_dir / "intermediate",
        ]
        
        for directory in directories:
            directory.mkdir(parents=True, exist_ok=True)
    
    def load_evaluation_dataset(self) -> dict[str, t.Any]:
        """Load the comprehensive evaluation dataset."""
        print(f"Loading evaluation dataset from {self.config.eval_dataset_path}")
        
        with open(self.config.eval_dataset_path) as f:
            dataset = json.load(f)
        
        self.evaluation_dataset = dataset
        
        # Print dataset summary
        conditions = dataset.get("conditions", {})
        print(f"Loaded evaluation dataset with {len(conditions)} conditions:")
        for condition_name, condition_data in conditions.items():
            if isinstance(condition_data, list):
                print(f"  {condition_name}: {len(condition_data)} models")
            elif isinstance(condition_data, dict):
                total_models = sum(len(models) for models in condition_data.values())
                print(f"  {condition_name}: {len(condition_data)} configs, {total_models} total models")
        
        return dataset
    
    def discover_and_validate_checkpoints(self) -> list[ModelMetadata]:
        """Discover available checkpoints and validate completeness."""
        print("Discovering model checkpoints...")
        metadata_list = self.checkpoint_manager.discover_checkpoints()
        
        if not metadata_list:
            raise RuntimeError("No valid checkpoints found")
        
        # Validate completeness
        validation_results = self.checkpoint_manager.validate_checkpoint_completeness(metadata_list)
        
        print(f"\nCheckpoint Validation:")
        print(f"  Total checkpoints: {validation_results['total_checkpoints']}")
        print(f"  Configs found: {len(validation_results['configs_found'])}")
        
        if validation_results['missing_configs']:
            warnings.warn(f"Missing configs: {validation_results['missing_configs']}")
        
        if validation_results['incomplete_diversity']:
            print("  Incomplete diversity coverage:")
            for config, missing in validation_results['incomplete_diversity']:
                print(f"    {config}: missing {missing}")
        
        return metadata_list
    
    def run_comprehensive_evaluation(self) -> dict[str, t.Any]:
        """Run the complete evaluation pipeline."""
        start_time = datetime.now()
        print("=" * 80)
        print("STARTING COMPREHENSIVE ICL EVALUATION")
        print("=" * 80)
        print(f"Start time: {start_time}")
        print(f"Output directory: {self.config.output_dir}")
        print()
        
        # Load evaluation dataset
        eval_dataset = self.load_evaluation_dataset()
        
        # Discover checkpoints
        checkpoint_metadata = self.discover_and_validate_checkpoints()
        
        # Create evaluation manifest
        manifest = create_evaluation_manifest(self.config, start_time)
        manifest_path = self.config.output_dir / "metadata" / "experiment_manifest.json"
        with open(manifest_path, "w") as f:
            json.dump(manifest, f, indent=2)
        
        # Save model registry
        model_registry_df = DataSchemaManager.metadata_to_dataframe(checkpoint_metadata)
        registry_path = self.config.output_dir / "metadata" / "model_registry.parquet"
        model_registry_df.to_parquet(registry_path, index=False)
        print(f"Saved model registry: {registry_path}")
        
        # Run evaluations
        all_results = []
        attention_records = []
        
        total_models = len(checkpoint_metadata)
        
        for model_idx, metadata in enumerate(tqdm(checkpoint_metadata, desc="Evaluating models")):
            print(f"\n[{model_idx + 1}/{total_models}] Evaluating {metadata.model_id}")
            
            try:
                # Load model
                model, tokenizer = self.checkpoint_manager.load_model_checkpoint(metadata)
                
                # Evaluate on all conditions
                model_results, model_attention = self._evaluate_single_model(
                    model, tokenizer, metadata, eval_dataset
                )
                
                all_results.extend(model_results)
                attention_records.extend(model_attention)
                
                # Save intermediate results periodically
                if self.config.save_intermediate and (model_idx + 1) % 5 == 0:
                    self._save_intermediate_results(all_results, attention_records)
                
                # Clear model from memory
                del model, tokenizer
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                
            except Exception as e:
                warnings.warn(f"Failed to evaluate model {metadata.model_id}: {e}")
                continue
        
        # Save final results
        print("\nSaving final results...")
        results_summary = self._save_final_results(
            all_results, attention_records, checkpoint_metadata
        )
        
        # Update manifest
        end_time = datetime.now()
        manifest["end_time"] = end_time.isoformat()
        manifest["status"] = "completed"
        manifest["results_summary"] = results_summary
        
        with open(manifest_path, "w") as f:
            json.dump(manifest, f, indent=2)
        
        print("=" * 80)
        print("COMPREHENSIVE EVALUATION COMPLETED")
        print("=" * 80)
        print(f"End time: {end_time}")
        print(f"Duration: {end_time - start_time}")
        print(f"Total evaluations: {len(all_results)}")
        print(f"Results saved to: {self.config.output_dir}")
        
        return {
            "start_time": start_time,
            "end_time": end_time,
            "total_evaluations": len(all_results),
            "total_models": len(checkpoint_metadata),
            "results_summary": results_summary,
            "output_dir": self.config.output_dir,
        }
    
    def _evaluate_single_model(
        self,
        model: t.Any,
        tokenizer: t.Any,
        metadata: ModelMetadata,
        eval_dataset: dict[str, t.Any]
    ) -> tuple[list[ICLPerformanceRecord], list[AttentionRecord]]:
        """Evaluate a single model on all evaluation conditions."""
        results = []
        attention_records = []
        
        # Get model's training configuration
        model_config = (metadata.config_L, metadata.config_m)
        
        # Evaluate on all transfer conditions
        for transfer_condition in self.config.transfer_conditions:
            condition_results, condition_attention = self._evaluate_transfer_condition(
                model, tokenizer, metadata, eval_dataset, transfer_condition, model_config
            )
            results.extend(condition_results)
            attention_records.extend(condition_attention)
        
        return results, attention_records
    
    def _evaluate_transfer_condition(
        self,
        model: t.Any,
        tokenizer: t.Any,
        metadata: ModelMetadata,
        eval_dataset: dict[str, t.Any],
        transfer_condition: TransferCondition,
        model_config: tuple[int, int]
    ) -> tuple[list[ICLPerformanceRecord], list[AttentionRecord]]:
        """Evaluate model on a specific transfer condition."""
        results = []
        attention_records = []
        
        # Get appropriate evaluation sequences based on transfer condition
        eval_sequences = self._get_sequences_for_condition(
            eval_dataset, transfer_condition, model_config
        )
        
        if not eval_sequences:
            warnings.warn(f"No sequences found for {transfer_condition} condition")
            return results, attention_records
        
        # Limit sequences if specified
        if self.config.max_sequences_per_condition > 0:
            eval_sequences = eval_sequences[:self.config.max_sequences_per_condition]
        
        # Evaluate across all context sizes and control types
        for context_size in self.config.context_sizes:
            for control_type in self.config.control_types:
                sequences = self._prepare_sequences_for_evaluation(
                    eval_sequences, context_size, control_type
                )
                
                for seq_idx, sequence in enumerate(sequences):
                    try:
                        # Evaluate sequence
                        is_correct, attention_data = self.evaluation_engine.evaluate_icl_sequence(
                            model, tokenizer, sequence, 
                            capture_attention=self.config.capture_attention
                        )
                        
                        # Create performance record
                        target_config = self._get_target_config(sequence, transfer_condition, model_config)
                        
                        record = ICLPerformanceRecord(
                            model_id=metadata.model_id,
                            config_L=metadata.config_L,
                            config_m=metadata.config_m,
                            n_train=metadata.n_train,
                            checkpoint_step=metadata.checkpoint_step,
                            context_size=context_size,
                            transfer_condition=transfer_condition,
                            target_config_L=target_config[0],
                            target_config_m=target_config[1],
                            accuracy=float(is_correct),
                            sequence_id=seq_idx,
                            control_type=control_type,
                            evaluation_timestamp=datetime.now(),
                            num_sequences=1,
                            num_correct=int(is_correct),
                        )
                        
                        results.append(record)
                        
                        # Store attention data if captured
                        if attention_data and self.config.capture_attention:
                            for attention_key, attention_matrix in attention_data.items():
                                layer_idx, head_idx = self._parse_attention_key(attention_key)
                                
                                attention_record = AttentionRecord(
                                    model_id=metadata.model_id,
                                    layer_idx=layer_idx,
                                    head_idx=head_idx,
                                    context_size=context_size,
                                    sequence_id=seq_idx,
                                    attention_matrix=attention_matrix,
                                    evaluation_timestamp=datetime.now(),
                                )
                                
                                attention_records.append(attention_record)
                        
                    except Exception as e:
                        warnings.warn(f"Failed to evaluate sequence {seq_idx}: {e}")
                        continue
        
        return results, attention_records
    
    def _get_sequences_for_condition(
        self,
        eval_dataset: dict[str, t.Any],
        transfer_condition: TransferCondition,
        model_config: tuple[int, int]
    ) -> list[dict[str, t.Any]]:
        """Get evaluation sequences for a specific transfer condition."""
        conditions = eval_dataset.get("conditions", {})
        
        if transfer_condition == "within_config":
            # Use within-config evaluation data
            within_config_data = conditions.get("within_config", [])
            for model_data in within_config_data:
                if tuple(model_data["config"]) == model_config:
                    sequences = []
                    for k_sequences in model_data["sequences"].values():
                        sequences.extend(k_sequences)
                    return sequences
        
        elif transfer_condition in ["cross_L", "cross_m", "cross_config"]:
            # Use appropriate transfer condition data
            condition_key = {
                "cross_L": "depth_transfer",
                "cross_m": "synonym_transfer", 
                "cross_config": "full_transfer"
            }[transfer_condition]
            
            transfer_data = conditions.get(condition_key, {})
            sequences = []
            
            for config_key, config_models in transfer_data.items():
                for model_data in config_models:
                    for k_sequences in model_data["sequences"].values():
                        sequences.extend(k_sequences)
            
            return sequences
        
        return []
    
    def _prepare_sequences_for_evaluation(
        self,
        sequences: list[dict[str, t.Any]],
        context_size: int,
        control_type: ControlType
    ) -> list[dict[str, t.Any]]:
        """Prepare sequences for evaluation with specified context size and control type."""
        prepared_sequences = []
        
        for sequence in sequences:
            # Filter by context size
            if sequence.get("context_size") == context_size:
                if control_type == "normal":
                    prepared_sequences.append(sequence)
                elif control_type == "shuffled_context":
                    # Create shuffled version
                    shuffled_seq = self._create_shuffled_sequence(sequence)
                    prepared_sequences.append(shuffled_seq)
                elif control_type == "random_context":
                    # Create random context version
                    random_seq = self._create_random_context_sequence(sequence, sequences)
                    if random_seq:
                        prepared_sequences.append(random_seq)
        
        return prepared_sequences
    
    def _create_shuffled_sequence(self, sequence: dict[str, t.Any]) -> dict[str, t.Any]:
        """Create a sequence with shuffled context order."""
        import random
        shuffled_seq = sequence.copy()
        
        # Shuffle context pairs
        context_pairs = list(zip(
            shuffled_seq["context_features"], 
            shuffled_seq["context_labels"],
            strict=False
        ))
        random.shuffle(context_pairs)
        
        shuffled_seq["context_features"] = [pair[0] for pair in context_pairs]
        shuffled_seq["context_labels"] = [pair[1] for pair in context_pairs]
        
        return shuffled_seq
    
    def _create_random_context_sequence(
        self, 
        sequence: dict[str, t.Any], 
        all_sequences: list[dict[str, t.Any]]
    ) -> dict[str, t.Any] | None:
        """Create a sequence with random context from other sequences."""
        import random
        
        # Find other sequences with same context size
        same_k_sequences = [
            seq for seq in all_sequences 
            if seq.get("context_size") == sequence.get("context_size")
            and seq != sequence
        ]
        
        if len(same_k_sequences) < sequence.get("context_size", 0):
            return None
        
        # Sample random contexts
        random_contexts = random.sample(same_k_sequences, sequence.get("context_size", 0))
        
        random_seq = sequence.copy()
        random_seq["context_features"] = [ctx["query_features"] for ctx in random_contexts]
        random_seq["context_labels"] = [ctx["query_label"] for ctx in random_contexts]
        
        return random_seq
    
    def _get_target_config(
        self,
        sequence: dict[str, t.Any],
        transfer_condition: TransferCondition,
        model_config: tuple[int, int]
    ) -> tuple[int, int]:
        """Get target configuration for the sequence."""
        # For within-config, target is same as model config
        if transfer_condition == "within_config":
            return model_config
        
        # For transfer conditions, try to infer from sequence metadata
        # This would need to be stored in the evaluation dataset
        # For now, return model config as fallback
        return model_config
    
    def _parse_attention_key(self, attention_key: str) -> tuple[int, int]:
        """Parse layer and head indices from attention key."""
        # Expected format: "layer_{layer_idx}_head_{head_idx}"
        parts = attention_key.split("_")
        layer_idx = int(parts[1])
        head_idx = int(parts[3])
        return layer_idx, head_idx
    
    def _save_intermediate_results(
        self,
        results: list[ICLPerformanceRecord],
        attention_records: list[AttentionRecord]
    ) -> None:
        """Save intermediate results to disk."""
        if results:
            df = DataSchemaManager.records_to_dataframe(results)
            intermediate_path = self.config.output_dir / "intermediate" / "partial_results.parquet"
            df.to_parquet(intermediate_path, index=False)
        
        if attention_records and self.config.capture_attention:
            self._save_attention_data(attention_records, intermediate=True)
    
    def _save_final_results(
        self,
        results: list[ICLPerformanceRecord],
        attention_records: list[AttentionRecord],
        checkpoint_metadata: list[ModelMetadata]
    ) -> dict[str, t.Any]:
        """Save final evaluation results."""
        # Save ICL performance results
        if results:
            df = DataSchemaManager.records_to_dataframe(results)
            results_path = self.config.output_dir / "raw_evaluations" / "icl_performance.parquet"
            df.to_parquet(results_path, index=False)
            print(f"Saved ICL performance results: {results_path} ({len(results)} records)")
        
        # Save attention data
        if attention_records and self.config.capture_attention:
            self._save_attention_data(attention_records)
            print(f"Saved attention data: {len(attention_records)} records")
        
        # Generate and save aggregated metrics
        aggregated_metrics = self._compute_aggregated_metrics(results)
        if aggregated_metrics:
            metrics_path = self.config.output_dir / "intermediate" / "aggregated_metrics.parquet"
            aggregated_metrics.to_parquet(metrics_path, index=False)
            print(f"Saved aggregated metrics: {metrics_path}")
        
        return {
            "total_evaluations": len(results),
            "total_models": len(checkpoint_metadata),
            "total_attention_records": len(attention_records),
            "unique_model_configs": len(set((m.config_L, m.config_m) for m in checkpoint_metadata)),
            "context_sizes_evaluated": list(set(r.context_size for r in results)),
            "transfer_conditions_evaluated": list(set(r.transfer_condition for r in results)),
        }
    
    def _save_attention_data(
        self, 
        attention_records: list[AttentionRecord], 
        intermediate: bool = False
    ) -> None:
        """Save attention data to individual files."""
        attention_dir = self.config.output_dir / "raw_evaluations" / "attention_data"
        if intermediate:
            attention_dir = attention_dir / "intermediate"
            attention_dir.mkdir(exist_ok=True)
        
        # Group by model for efficient storage
        by_model = defaultdict(list)
        for record in attention_records:
            by_model[record.model_id].append(record)
        
        for model_id, model_records in by_model.items():
            model_dir = attention_dir / model_id
            model_dir.mkdir(exist_ok=True)
            
            for record in model_records:
                filename = record.get_filename()
                filepath = model_dir / filename
                
                np.savez_compressed(
                    filepath,
                    attention_matrix=record.attention_matrix,
                    metadata={
                        "model_id": record.model_id,
                        "layer_idx": record.layer_idx,
                        "head_idx": record.head_idx,
                        "context_size": record.context_size,
                        "sequence_id": record.sequence_id,
                        "timestamp": record.evaluation_timestamp.isoformat(),
                    }
                )
    
    def _compute_aggregated_metrics(
        self, 
        results: list[ICLPerformanceRecord]
    ) -> pd.DataFrame | None:
        """Compute aggregated metrics from evaluation results."""
        if not results:
            return None
        
        # Convert to DataFrame for easier aggregation
        df = DataSchemaManager.records_to_dataframe(results)
        
        # Group by model and compute aggregated metrics
        aggregated = []
        
        groupby_cols = ["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]
        
        for name, group in df.groupby(groupby_cols):
            model_id, config_L, config_m, n_train, checkpoint_step = name
            
            # Compute emergence threshold (minimum k for >0.5 accuracy on normal sequences)
            normal_group = group[group["control_type"] == "normal"]
            emergence_threshold = self._compute_emergence_threshold(normal_group)
            
            # Compute max accuracy across all context sizes
            max_accuracy = normal_group["accuracy"].max() if len(normal_group) > 0 else 0.0
            
            # Compute transfer degradation
            within_config_acc = normal_group[
                normal_group["transfer_condition"] == "within_config"
            ]["accuracy"].mean()
            
            transfer_acc = normal_group[
                normal_group["transfer_condition"] != "within_config"
            ]["accuracy"].mean()
            
            transfer_degradation = within_config_acc - transfer_acc if not pd.isna(within_config_acc) and not pd.isna(transfer_acc) else 0.0
            
            # Compute baseline gap (normal vs shuffled/random)
            normal_acc = group[group["control_type"] == "normal"]["accuracy"].mean()
            control_acc = group[group["control_type"] != "normal"]["accuracy"].mean()
            baseline_gap = normal_acc - control_acc if not pd.isna(normal_acc) and not pd.isna(control_acc) else 0.0
            
            aggregated.append({
                "model_id": model_id,
                "config_L": config_L,
                "config_m": config_m,
                "n_train": n_train,
                "checkpoint_step": checkpoint_step,
                "emergence_threshold": emergence_threshold,
                "max_context_accuracy": max_accuracy,
                "transfer_degradation": transfer_degradation,
                "baseline_gap": baseline_gap,
                "total_evaluations": len(group),
            })
        
        return pd.DataFrame(aggregated)
    
    def _compute_emergence_threshold(self, group: pd.DataFrame) -> float:
        """Compute emergence threshold (minimum k for >0.5 accuracy)."""
        threshold = 0.5
        
        # Group by context size and compute mean accuracy
        by_context = group.groupby("context_size")["accuracy"].mean().sort_index()
        
        # Find first context size with accuracy > threshold
        for context_size, accuracy in by_context.items():
            if accuracy > threshold:
                return float(context_size)
        
        # If no context size achieves threshold, return max context size + 1
        return float(max(group["context_size"]) + 1) if len(group) > 0 else float('inf')

# EvaluationCoordinator

In [12]:
"""Main coordinator for the comprehensive evaluation pipeline."""
import logging

class EvaluationCoordinator:
    """Coordinates the complete evaluation pipeline."""
    
    def __init__(self, config: EvaluationConfig):
        """Initialize evaluation coordinator."""
        self.config = config
        self.setup_logging()
        
    def setup_logging(self) -> None:
        """Setup logging for the evaluation pipeline."""
        log_dir = self.config.output_dir / "logs"
        log_dir.mkdir(parents=True, exist_ok=True)
        
        log_file = log_dir / f"evaluation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_file),
                logging.StreamHandler()
            ]
        )
        
        self.logger = logging.getLogger(__name__)
        self.logger.info(f"Evaluation coordinator initialized. Logs: {log_file}")
    
    def run_evaluation_pipeline(self) -> dict[str, t.Any]:
        """Run the complete evaluation pipeline."""
        self.logger.info("Starting comprehensive ICL evaluation pipeline")
        
        try:
            # Initialize evaluator
            evaluator = ComprehensiveEvaluator(self.config)
            
            # Run comprehensive evaluation
            results = evaluator.run_comprehensive_evaluation()
            
            self.logger.info("Evaluation pipeline completed successfully")
            return results
            
        except Exception as e:
            self.logger.error(f"Evaluation pipeline failed: {e}")
            raise
    
    def validate_setup(self) -> bool:
        """Validate that all required components are available."""
        self.logger.info("Validating evaluation setup...")
        
        try:
            # Validate configuration
            self.config.validate()
            
            # Check if evaluation dataset exists
            if not self.config.eval_dataset_path.exists():
                raise FileNotFoundError(f"Evaluation dataset not found: {self.config.eval_dataset_path}")
            
            # Check if checkpoint directories exist
            missing_dirs = []
            for checkpoint_dir in self.config.checkpoint_base_dirs:
                if not checkpoint_dir.exists():
                    missing_dirs.append(checkpoint_dir)
            
            if missing_dirs:
                raise FileNotFoundError(f"Missing checkpoint directories: {missing_dirs}")
            
            # Validate evaluation dataset format
            self._validate_evaluation_dataset()
            
            self.logger.info("Setup validation completed successfully")
            return True
            
        except Exception as e:
            self.logger.error(f"Setup validation failed: {e}")
            return False
    
    def _validate_evaluation_dataset(self) -> None:
        """Validate the evaluation dataset format."""
        with open(self.config.eval_dataset_path) as f:
            dataset = json.load(f)
        
        # Check required top-level keys
        required_keys = ["metadata", "conditions"]
        for key in required_keys:
            if key not in dataset:
                raise ValueError(f"Missing required key in evaluation dataset: {key}")
        
        # Check conditions structure
        conditions = dataset["conditions"]
        expected_conditions = ["within_config", "depth_transfer", "synonym_transfer", "full_transfer"]
        
        for condition in expected_conditions:
            if condition not in conditions:
                self.logger.warning(f"Missing evaluation condition: {condition}")
        
        self.logger.info("Evaluation dataset format validation passed")
    
    def generate_evaluation_summary(self, results: dict[str, t.Any]) -> Path:
        """Generate a comprehensive summary of evaluation results."""
        summary_path = self.config.output_dir / "evaluation_summary.json"
        
        summary = {
            "evaluation_config": {
                "context_sizes": self.config.context_sizes,
                "transfer_conditions": self.config.transfer_conditions,
                "control_types": self.config.control_types,
                "target_configs": self.config.target_configs,
                "diversity_levels": self.config.diversity_levels,
                "model_types": self.config.model_types,
                "device": self.config.device,
                "capture_attention": self.config.capture_attention,
            },
            "results": results,
            "output_files": {
                "icl_performance": "raw_evaluations/icl_performance.parquet",
                "model_registry": "metadata/model_registry.parquet",
                "aggregated_metrics": "intermediate/aggregated_metrics.parquet",
                "attention_data": "raw_evaluations/attention_data/",
                "logs": "logs/",
            },
            "next_steps": {
                "rq1_emergence": "Run analysis/rq1_emergence_analyzer.py",
                "rq2_scaling": "Run analysis/rq2_scaling_analyzer.py", 
                "rq3_mechanistic": "Run analysis/rq3_mechanistic_analyzer.py",
                "rq4_transfer": "Run analysis/rq4_transfer_analyzer.py",
                "rq5_diversity": "Run analysis/rq5_diversity_analyzer.py",
                "rq6_comparative": "Run analysis/rq6_comparative_analyzer.py",
            }
        }
        
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2, default=str)
        
        self.logger.info(f"Evaluation summary saved: {summary_path}")
        return summary_path

def create_evaluation_config(
    checkpoint_dirs: list[Path],
    eval_dataset_path: Path,
    output_dir: Path,
    target_configs: list[tuple[int, int]],
    **kwargs
) -> EvaluationConfig:
    """Helper function to create evaluation configuration."""
    
    config = EvaluationConfig(
        checkpoint_base_dirs=checkpoint_dirs,
        eval_dataset_path=eval_dataset_path,
        output_dir=output_dir,
        target_configs=target_configs,
        **kwargs
    )
    
    return config

def run_comprehensive_evaluation(
    checkpoint_dirs: list[str | Path],
    eval_dataset_path: str | Path,
    output_dir: str | Path,
    target_configs: list[tuple[int, int]],
    **kwargs
) -> dict[str, t.Any]:
    """High-level function to run comprehensive evaluation."""
    
    # Convert paths
    checkpoint_dirs = [Path(d) for d in checkpoint_dirs]
    eval_dataset_path = Path(eval_dataset_path)
    output_dir = Path(output_dir)
    
    # Create configuration
    config = create_evaluation_config(
        checkpoint_dirs=checkpoint_dirs,
        eval_dataset_path=eval_dataset_path,
        output_dir=output_dir,
        target_configs=target_configs,
        **kwargs
    )
    
    # Initialize coordinator
    coordinator = EvaluationCoordinator(config)
    
    # Validate setup
    if not coordinator.validate_setup():
        raise RuntimeError("Evaluation setup validation failed")
    
    # Run evaluation pipeline
    results = coordinator.run_evaluation_pipeline()
    
    # Generate summary
    summary_path = coordinator.generate_evaluation_summary(results)
    
    print(f"\nEvaluation completed successfully!")
    print(f"Results: {output_dir}")
    print(f"Summary: {summary_path}")
    
    return results

# Run eval

In [15]:
"""Example script to run Phase 1 comprehensive evaluation."""

from pathlib import Path
from datetime import datetime



def main():
    """Run comprehensive evaluation pipeline."""
    
    # Configuration
    checkpoint_dirs = [
        Path("checkpoints/causal_lm"),
        Path("checkpoints/mlm"),
    ]
    
    eval_dataset_path = data_dir /"eval/verified_transfer_evaluation_dataset.json"
    output_dir = data_dir / "raw_results"
    
    # Target configurations to evaluate (L, m pairs)
    target_configs = [
        (2, 2), (2, 3), (2, 4),
        (3, 2), (3, 3), (3, 4),
        (4, 2), (4, 3), (4, 4),
    ]
    
    # Evaluation parameters
    evaluation_params = {
        "context_sizes": [1, 2, 3, 4, 5, 6, 8],
        "diversity_levels": [8, 16, 32, 64, 128],
        "model_types": ["causal_lm", "mlm"],
        "device": "cuda",
        "batch_size": 16,
        "max_sequences_per_condition": 100,
        "capture_attention": True,
        "capture_representations": False,
        "save_intermediate": True,
        "overwrite_existing": False,
    }
    
    print("=" * 80)
    print("COMPREHENSIVE ICL EVALUATION PIPELINE")
    print("=" * 80)
    print(f"Checkpoint directories: {checkpoint_dirs}")
    print(f"Evaluation dataset: {eval_dataset_path}")
    print(f"Output directory: {output_dir}")
    print(f"Target configurations: {target_configs}")
    print(f"Context sizes: {evaluation_params['context_sizes']}")
    print(f"Diversity levels: {evaluation_params['diversity_levels']}")
    print(f"Device: {evaluation_params['device']}")
    print()
    
    # Validate paths exist
    for checkpoint_dir in checkpoint_dirs:
        if not checkpoint_dir.exists():
            print(f"WARNING: Checkpoint directory not found: {checkpoint_dir}")
    
    if not eval_dataset_path.exists():
        print(f"ERROR: Evaluation dataset not found: {eval_dataset_path}")
        return
    
    # Run evaluation
    try:
        start_time = datetime.now()
        
        results = run_comprehensive_evaluation(
            checkpoint_dirs=checkpoint_dirs,
            eval_dataset_path=eval_dataset_path,
            output_dir=output_dir,
            target_configs=target_configs,
            **evaluation_params
        )
        
        end_time = datetime.now()
        duration = end_time - start_time
        
        print("\n" + "=" * 80)
        print("EVALUATION COMPLETED SUCCESSFULLY")
        print("=" * 80)
        print(f"Duration: {duration}")
        print(f"Total evaluations: {results['total_evaluations']}")
        print(f"Total models: {results['total_models']}")
        print(f"Output directory: {results['output_dir']}")
        print()
        
        print("Next steps:")
        print("1. Run RQ1 emergence analysis: python -m analysis.rq1_emergence_analyzer")
        print("2. Run RQ2 scaling analysis: python -m analysis.rq2_scaling_analyzer")
        print("3. Run RQ3 mechanistic analysis: python -m analysis.rq3_mechanistic_analyzer")
        print("4. Run RQ4 transfer analysis: python -m analysis.rq4_transfer_analyzer")
        print("5. Run RQ5 diversity analysis: python -m analysis.rq5_diversity_analyzer")
        print("6. Run RQ6 comparative analysis: python -m analysis.rq6_comparative_analyzer")
        
    except Exception as e:
        print(f"\nERROR: Evaluation failed: {e}")
        raise

def test_configuration():
    """Test evaluation configuration without running full evaluation."""
    from data_collection.evaluation_coordinator import EvaluationCoordinator, create_evaluation_config
    
    checkpoint_dirs = [Path("checkpoints/causal_lm")]
    eval_dataset_path = Path("eval_data/verified_transfer_evaluation_dataset.json")
    output_dir = Path("results/test_evaluation")
    target_configs = [(2, 2), (3, 3)]
    
    config = create_evaluation_config(
        checkpoint_dirs=checkpoint_dirs,
        eval_dataset_path=eval_dataset_path,
        output_dir=output_dir,
        target_configs=target_configs,
        max_sequences_per_condition=10,  # Small number for testing
        capture_attention=False,  # Disable for faster testing
    )
    
    coordinator = EvaluationCoordinator(config)
    
    print("Testing evaluation configuration...")
    is_valid = coordinator.validate_setup()
    
    if is_valid:
        print("✓ Configuration is valid")
        
        # Test checkpoint discovery
        checkpoint_manager = CheckpointManager(config)
        metadata = checkpoint_manager.discover_checkpoints()
        
        print(f"✓ Discovered {len(metadata)} checkpoints")
        
        if metadata:
            print("Sample checkpoint metadata:")
            for i, meta in enumerate(metadata[:3]):
                print(f"  {i+1}. {meta.model_id} - L{meta.config_L}_m{meta.config_m}_ntrain{meta.n_train}")
    else:
        print("✗ Configuration validation failed")

if __name__ == "__main__":
    import argparse
    
    parser = argparse.ArgumentParser(description="Run comprehensive ICL evaluation")
    parser.add_argument("--test", action="store_true", help="Test configuration only")
    parser.add_argument("--device", default="cuda", help="Device to use (cuda/cpu)")
    parser.add_argument("--max-sequences", type=int, default=100, 
                       help="Maximum sequences per condition")
    
    args = parser.parse_args()
    
    if args.test:
        test_configuration()
    else:
        main()

usage: ipykernel_launcher.py [-h] [--test] [--device DEVICE]
                             [--max-sequences MAX_SEQUENCES]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/jliu/Library/Jupyter/runtime/kernel-v389c2952445eba1840dfdd071d9bc183b51b210e3.json


SystemExit: 2

/Users/jliu/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
